# 05 · Regression — California housing

**Dataset:** `data/housing_raw.csv` — California housing (20,640 rows) with a synthetic
`Neighbourhood`, `HouseType` and `Condition`, plus some injected mess.
**Target:** `MedianHouseValue` (median house value per block, in $100k).
**Covers:** the whole pipeline again, but for **regression** — where the metrics,
the target distribution and the diagnostics all differ.
**Time yourself:** ~45 minutes.

Classification questions ("what's the survival rate?") get replaced by
"what's the average price per neighbourhood?" — this is the dataset shape a
house-price interview actually uses.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

df = pd.read_csv('data/housing_raw.csv')
print(df.shape)
df.head()

**Column reference**

| column | meaning |
|---|---|
| `MedInc` | median household income in the block, in **dollars** |
| `HouseAge` | median house age in the block |
| `AveRooms` / `AveBedrms` | average rooms / bedrooms per household |
| `Population` | block population |
| `AveOccup` | average household occupancy |
| `Latitude` / `Longitude` | block centroid |
| `Neighbourhood`, `HouseType`, `Condition` | synthetic categoricals |
| `MedianHouseValue` | **target** — median house value, in $100k |

---

## Part A — Clean

### Q1. First look: shape, dtypes, missing values, duplicates. List every problem you find before fixing any of them.

<details><summary>hint</summary>

`.info()`, `.describe()`, `.duplicated().sum()`, `.isnull().mean()`. Look at `Condition.unique()` too.

</details>

In [ ]:
print(df.shape)
df.info()
display(df.describe().round(2))

print('\nduplicated rows:', df.duplicated().sum())
print('\nmissing %:')
print((df.isnull().mean() * 100).round(2)[lambda s: s > 0])
print('\nCondition levels:', df['Condition'].unique())
print('MedInc sample:', df['MedInc'].dropna().head(3).tolist())

# Problems:
#  1. MedInc is object -- "$38,462" strings, blanks for missing
#  2. 40 duplicated rows
#  3. HouseAge ~2% and AveRooms ~7% missing
#  4. Condition has inconsistent casing/whitespace ('good', 'GOOD', 'Good', ' good')

### Q2. Fix them all: drop duplicates, parse `MedInc` to numeric (mind the `$` and the thousands separators), normalise `Condition`.

<details><summary>hint</summary>

Same moves as notebook 01: strip separators, then `pd.to_numeric(..., errors='coerce')`; `.str.strip().str.lower()` for the categorical.

</details>

In [ ]:
df = df.drop_duplicates()

df['MedInc'] = pd.to_numeric(df['MedInc']
                             .str.replace('$', '', regex=False)
                             .str.replace(',', '', regex=False)
                             .str.strip(),
                             errors='coerce')
df['Condition'] = df['Condition'].str.strip().str.lower()

print(df.shape, '|', df['MedInc'].dtype)
print(df['Condition'].value_counts())
print('\nmissing %:')
print((df.isnull().mean() * 100).round(2)[lambda s: s > 0])

### Q3. Look hard at `MedianHouseValue`'s maximum and count how many rows sit exactly at it.
What has been done to this data, and what does it mean for your model?

<details><summary>hint</summary>

Sort the target descending and look at how many rows share the top value. It's not a coincidence.

</details>

In [ ]:
mx = df['MedianHouseValue'].max()
n_at_max = (df['MedianHouseValue'] == mx).sum()
print(f'max = {mx} | {n_at_max} rows ({n_at_max/len(df):.1%}) are exactly at it')

sns.histplot(df['MedianHouseValue'], bins=60)
plt.axvline(mx, color='r', ls='--')
plt.title('Target — note the spike at the cap')
plt.show()

# The target is CENSORED (top-coded) at 5.0 = $500k: every block worth more than $500k was
# recorded as exactly $500k. ~5% of rows. Consequences:
#  - The model cannot learn anything above the cap; it will systematically under-predict
#    expensive blocks, and residual plots will show a hard diagonal edge at the top.
#  - Those rows are not outliers to clip -- they're a measurement artefact.
#  - Options: model it as-is and caveat the top 5%; drop the capped rows (biases you toward
#    cheap areas); or treat it properly as a censored-regression problem (Tobit).
#  - Minimum viable answer in an interview: SAY IT OUT LOUD. Spotting the cap is the single
#    most common thing candidates miss on this dataset.

### Q4. Handle the missing `MedInc`, `HouseAge` and `AveRooms`. Choose mean or median per
column and justify with the skew. (Just do it globally for now — you'll redo it
leak-free in the pipeline later.)

<details><summary>hint</summary>

Check `.skew()` first. Rule of thumb: |skew| > 1 → median.

</details>

In [ ]:
print(df[['MedInc', 'HouseAge', 'AveRooms']].skew().round(2))

# MedInc skew ~1.65 and AveRooms ~21 (!) -- both clearly skewed, so both get the median; a
# mean on AveRooms would be dragged badly by the extreme blocks. HouseAge is ~0.06, i.e.
# near-symmetric, so mean or median are equivalent -- median for consistency.
for col in ['MedInc', 'HouseAge', 'AveRooms']:
    df[col] = df[col].fillna(df[col].median())

print('missing left:', df.isnull().sum().sum())

---

## Part B — EDA & the questions they actually ask

### Q5. **The classic warm-up:** what is the average house value per neighbourhood?
Sort descending, and include the count per group.

<details><summary>hint</summary>

`groupby('Neighbourhood')['MedianHouseValue'].agg(...)`. Always report the count next to the mean.

</details>

In [ ]:
by_hood = (df.groupby('Neighbourhood')['MedianHouseValue']
             .agg(n='count', avg_value='mean', median_value='median')
             .sort_values('avg_value', ascending=False))
display(by_hood.round(3))

# Values are in $100k, so multiply by 100,000 to speak in dollars.

### Q6. Follow-up they always ask: the **top 3 neighbourhoods by average value, but only
those with at least 500 blocks**, and show the average income alongside.

<details><summary>hint</summary>

Named aggregation `agg(n=('col','size'), avg=('col','mean'))`, then `.query()` to filter groups, then `.nlargest(3, 'avg_value')`.

</details>

In [ ]:
top = (df.groupby('Neighbourhood')
         .agg(n=('MedianHouseValue', 'size'),
              avg_value=('MedianHouseValue', 'mean'),
              avg_income=('MedInc', 'mean'))
         .query('n >= 500')
         .nlargest(3, 'avg_value'))
display(top.round(3))

### Q7. Cross two categoricals: average value by `Neighbourhood` × `Condition`, as a pivot
table. Does `Condition` matter? (Careful — you know how this column was made.)

<details><summary>hint</summary>

`pivot_table(index=..., columns=..., values=..., aggfunc='mean')`. Then ask whether the column-to-column variation is bigger than noise.

</details>

In [ ]:
pivot = df.pivot_table(index='Neighbourhood', columns='Condition',
                       values='MedianHouseValue', aggfunc='mean')
pivot = pivot[['poor', 'fair', 'good', 'excellent']]
display(pivot.round(3))

sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Avg value by neighbourhood x condition')
plt.show()

# The rows differ a lot, the columns barely at all. Condition was generated at RANDOM and
# is independent of the target -- it is pure noise, and the small variations you see are
# sampling error. This is the point of the question: a plausible-sounding categorical that
# carries zero signal. A good model should give it ~zero importance; if it doesn't, that's
# overfitting. Check features for signal before believing them.

### Q8. Which numeric feature correlates most with the target? Show a heatmap, then a
scatterplot of the strongest pair.

<details><summary>hint</summary>

`corr['MedianHouseValue'].sort_values()`. Sample before scattering 20k points — it's faster and less of an ink blob.

</details>

In [ ]:
num = df.select_dtypes('number')
corr = num.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

print(corr['MedianHouseValue'].drop('MedianHouseValue').sort_values(ascending=False).round(3))

sns.scatterplot(data=df.sample(3000, random_state=0), x='MedInc', y='MedianHouseValue',
                alpha=0.3)
plt.title('MedInc vs MedianHouseValue — note the horizontal line at the 5.0 cap')
plt.show()

# MedInc at ~0.67 dominates everything else (next best is rooms_per_person at ~0.38);
# income is basically the whole story. The scatter also shows the censoring as a hard
# horizontal stripe along the top.

### Q9. `AveRooms` has skew ~20. Look at its extreme values — error or real? Then decide
what to do and act.

<details><summary>hint</summary>

Look at `Population` and `AveOccup` on those rows before calling it an error. The clue is in the denominator.

</details>

In [ ]:
display(df.nlargest(5, 'AveRooms')[['AveRooms', 'AveBedrms', 'Population', 'AveOccup',
                                    'MedianHouseValue']])

q1, q3 = df['AveRooms'].quantile([0.25, 0.75])
iqr = q3 - q1
n_out = ((df['AveRooms'] < q1 - 1.5*iqr) | (df['AveRooms'] > q3 + 1.5*iqr)).sum()
print(f'{n_out} IQR outliers ({n_out/len(df):.1%})')

# 100+ rooms per household is not a mansion -- these blocks have tiny populations, so the
# ratio's denominator collapses. It's a real-but-degenerate statistic, not a typo.
# Winsorize rather than delete: keep the row (its other columns are fine) but stop the
# extreme value from dominating a distance- or scale-based model.
lo, hi = df['AveRooms'].quantile([0.01, 0.99])
df['AveRooms'] = df['AveRooms'].clip(lo, hi)
print('skew after clipping:', round(df['AveRooms'].skew(), 2))

---

## Part C — Features

### Q10. Engineer three features that should beat the raw columns:
`rooms_per_person`, `bedrooms_ratio` (bedrooms as a share of rooms), and
`households` (how many households are in the block).
Check their correlation with the target.

One of these three turns out to be nearly useless. Which, and why?

<details><summary>hint</summary>

Ratios of existing columns: `bedrooms_ratio = AveBedrms / AveRooms`, `households = Population / AveOccup`. Then compare the two ratios against the raw count and ask what each actually measures.

</details>

In [ ]:
df['rooms_per_person'] = df['AveRooms'] / df['AveOccup']
df['bedrooms_ratio'] = df['AveBedrms'] / df['AveRooms']
df['households'] = df['Population'] / df['AveOccup']

new = ['rooms_per_person', 'bedrooms_ratio', 'households']
print(df[new + ['MedianHouseValue']].corr()['MedianHouseValue'].round(3))

# rooms_per_person  +0.38  -- the best of the three: space per person is a wealth proxy.
# bedrooms_ratio    -0.22  -- a high bedroom share means small, cramped units.
# households        +0.07  -- nearly useless. It's a raw SIZE measure: a block having many
#                            households says nothing about whether they're rich.
#
# The pattern: the two RATIOS work, the raw COUNT doesn't. Ratios are scale-free -- "5
# rooms" means different things for a 2-person and a 6-person household, and dividing
# removes that confound. Counts mostly encode how big the block is, which isn't the
# question. When engineering features, prefer the normalised version of a quantity.
#
# WATCH OUT for the algebra trap: a tempting "pop_per_household = Population / (Population
# / AveOccup)" simplifies to exactly AveOccup -- a duplicate column with a new name, zero
# new information. Always simplify the expression before you trust the feature.

### Q11. The target is right-skewed. Compare `MedianHouseValue` against `log1p` of it: plot
both and print both skews. Would you train on the log target? What breaks if you do?

<details><summary>hint</summary>

`np.log1p` / `np.expm1`. The trap is in what happens to your reported metric.

</details>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(df['MedianHouseValue'], bins=50, ax=axes[0]).set_title('raw')
sns.histplot(np.log1p(df['MedianHouseValue']), bins=50, ax=axes[1]).set_title('log1p')
plt.tight_layout(); plt.show()

print('skew raw :', round(df['MedianHouseValue'].skew(), 2))
print('skew log :', round(np.log1p(df['MedianHouseValue']).skew(), 2))

# The log is more symmetric and would help a linear model (it assumes roughly normal
# errors, and it turns "price is multiplicative in income" into something additive).
# What breaks: your metric changes meaning. An RMSE of 0.3 in log space is not $30k -- it's
# roughly a 30% error. You must np.expm1() the predictions back before reporting RMSE, and
# that back-transform is biased low (Jensen's inequality) unless you correct for it.
# Here the censoring at 5.0 blunts the benefit anyway. Verdict: not worth it for a tree
# model; worth testing for the linear one.

---

## Part D — Model

### Q12. Split 80/20 with `random_state=42`. Then answer: why is there no `stratify=` here,
and is a random split even appropriate for this data?

<details><summary>hint</summary>

`stratify` needs discrete classes. Then think about what Latitude/Longitude imply about neighbouring rows.

</details>

In [ ]:
from sklearn.model_selection import train_test_split

TARGET = 'MedianHouseValue'
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

# No stratify: it stratifies on class labels, and a continuous target has none. (If you
# wanted balanced price bands across the splits you'd bin the target and stratify on the
# bin -- a real technique when the target is very skewed.)
# Is random appropriate? It's the honest default here since there's no time dimension. BUT
# the rows are spatially autocorrelated: neighbouring blocks are nearly identical, so a
# random split puts a block's near-twin in train and in test. That inflates the test score
# versus how the model would do on a genuinely new region. A grouped/spatial split
# (GroupKFold on Neighbourhood) is the more honest evaluation.

### Q13. Build a `ColumnTransformer` + `Pipeline` and fit a `LinearRegression` baseline.
Report RMSE, MAE and R² on test.

<details><summary>hint</summary>

Same shape as notebook 03. RMSE = `mean_squared_error(...) ** 0.5`. Translate it into dollars — the target is in $100k.

</details>

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

NUMERIC = X.select_dtypes('number').columns.tolist()
CATEGORICAL = ['Neighbourhood', 'HouseType', 'Condition']

pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), NUMERIC),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), CATEGORICAL),
])

def evaluate(model, name):
    pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    print(f'{name:22s} RMSE {rmse:.3f} (${rmse*100_000:,.0f}) | '
          f'MAE {mean_absolute_error(y_test, pred):.3f} | R2 {r2_score(y_test, pred):.3f}')
    return rmse

lin = Pipeline([('pre', pre), ('reg', LinearRegression())]).fit(X_train, y_train)
evaluate(lin, 'LinearRegression')

# R2 ~0.66: income and location explain about two thirds of the variance. RMSE ~0.67 means
# a typical error of ~$67k on a target whose median is ~$180k -- worth saying in dollars,
# not in model units. Nobody outside the team knows what "RMSE 0.67" means.

### Q14. Explain the difference between RMSE and MAE **using your own two numbers above**.
Which would you report to a business stakeholder pricing houses?

<details><summary>hint</summary>

RMSE squares the errors before averaging, so it is dominated by the worst ones. Which one is a sentence a human can act on?

</details>

In [ ]:
pred = lin.predict(X_test)
err = np.abs(y_test - pred)
print('MAE :', round(err.mean(), 3))
print('RMSE:', round((((y_test - pred) ** 2).mean()) ** 0.5, 3))
print('90th pct abs error:', round(err.quantile(0.9), 3))

# MAE ~0.48, RMSE ~0.67. RMSE > MAE always, and the size of that gap measures how uneven
# the errors are: squaring means one $200k miss counts as much as sixteen $50k misses. A
# ~40% gap here says the error distribution has a heavy tail -- confirmed by the 90th
# percentile absolute error of ~1.0 ($100k), double the typical miss. Much of that tail is
# the censored blocks the model cannot get right.
# To a stakeholder: MAE. "We're off by about $48k on a typical house" is a sentence they
# can act on; RMSE is not in any unit a human owns. Report MAE as the headline and RMSE
# alongside it if large errors are disproportionately costly (they are in lending -- one
# badly mispriced house can outweigh a hundred slightly-off ones).

### Q15. Fit a `RandomForestRegressor` and a `HistGradientBoostingRegressor`. Compare all three
models. How much did the non-linear models buy you over linear?

<details><summary>hint</summary>

`HistGradientBoostingRegressor` is sklearn's built-in LightGBM-alike — fast and usually the strongest thing you have without installing xgboost.

</details>

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

rf = Pipeline([('pre', pre),
               ('reg', RandomForestRegressor(n_estimators=200, random_state=42,
                                             n_jobs=-1))]).fit(X_train, y_train)
hgb = Pipeline([('pre', pre),
                ('reg', HistGradientBoostingRegressor(random_state=42))]).fit(X_train, y_train)

evaluate(lin, 'LinearRegression')
evaluate(rf, 'RandomForest')
evaluate(hgb, 'HistGradientBoosting')

# R2 goes ~0.66 (linear) -> ~0.81 (RF) -> ~0.83 (HGB), and RMSE ~0.67 -> ~0.48, i.e. the
# typical error drops from ~$67k to ~$48k. That's a big, real jump, unlike Titanic: value is
# genuinely non-linear in location (lat/long interact -- "coastal AND north"), and trees
# carve that up while a linear model can only tilt a plane through it.
# HistGradientBoosting is the modern default for tabular data: LightGBM-style histogram
# binning, handles NaNs natively, and it's fast on 20k rows.

### Q16. Cross-validate the winner with 5-fold `KFold` (`scoring='neg_root_mean_squared_error'`).
Why `neg_`?

<details><summary>hint</summary>

sklearn scorers are always 'higher is better'. RMSE isn't, so it's negated.

</details>

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = -cross_val_score(hgb, X_train, y_train, cv=kf,
                          scoring='neg_root_mean_squared_error', n_jobs=-1)
print('fold RMSEs:', scores.round(3))
print(f'CV RMSE: {scores.mean():.3f} +/- {scores.std():.3f}')

# sklearn's convention is that higher = better for every scorer, so anything that's really
# a loss gets negated. Flip the sign back to read it. Plain KFold (not Stratified) because
# there are no classes to preserve.

---

## Part E — Diagnose

### Q17. Plot predicted vs actual, and residuals vs predicted, for the boosting model.
Name **two** specific pathologies visible in these plots.

<details><summary>hint</summary>

Residuals should look like a shapeless cloud around zero. Anything with structure — a diagonal edge, a widening fan — is the model telling you something it can't handle.

</details>

In [ ]:
pred = hgb.predict(X_test)
resid = y_test - pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(pred, y_test, alpha=0.2, s=8)
axes[0].plot([0, 5.2], [0, 5.2], 'r--')
axes[0].set_xlabel('predicted'); axes[0].set_ylabel('actual')
axes[0].set_title('Predicted vs actual')

axes[1].scatter(pred, resid, alpha=0.2, s=8)
axes[1].axhline(0, color='r', ls='--')
axes[1].set_xlabel('predicted'); axes[1].set_ylabel('residual')
axes[1].set_title('Residuals vs predicted')
plt.tight_layout(); plt.show()

# 1. The censoring: a dense horizontal band of actuals sitting at exactly 5.0 while
#    predictions spread from ~2 to ~5. In the residual plot it's a hard descending
#    diagonal edge -- the model literally cannot win on those rows.
# 2. Heteroskedasticity: the residual cloud fans out as predictions grow. Errors on
#    expensive blocks are much larger in absolute terms than on cheap ones, so a single
#    global RMSE misrepresents both ends. This is the argument for a log target, or for
#    reporting a percentage error instead.
# (Bonus: residuals skew slightly negative at the top -- systematic under-prediction of
#  expensive blocks, a direct consequence of #1.)

### Q18. Check train vs test R² for the random forest. Over- or underfitting? Then fix it and
show the fix worked.

<details><summary>hint</summary>

Compare `r2_score(y_train, model.predict(X_train))` against test. A gap over ~0.1 with the train score near 1.0 is the tell.

</details>

In [ ]:
from sklearn.metrics import r2_score

print('RF   train R2:', round(r2_score(y_train, rf.predict(X_train)), 3),
      '| test R2:', round(r2_score(y_test, rf.predict(X_test)), 3))

rf_fixed = Pipeline([('pre', pre),
                     ('reg', RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                                   max_features='sqrt', random_state=42,
                                                   n_jobs=-1))]).fit(X_train, y_train)
print('RF+  train R2:', round(r2_score(y_train, rf_fixed.predict(X_train)), 3),
      '| test R2:', round(r2_score(y_test, rf_fixed.predict(X_test)), 3))

# Overfitting, badly: train R2 ~0.97 vs test ~0.81. Fully-grown trees memorise -- with
# min_samples_leaf=1 a leaf can hold a single block.
# Raising min_samples_leaf shrinks the gap. Note the test score barely moves: a random
# forest's averaging already absorbs most of the memorisation, which is exactly why it's a
# safe default. The gap is still worth closing -- it's free variance reduction and a
# smaller model.

### Q19. Get permutation importance on the test set. Does `Condition` show up as unimportant,
as you predicted in Part B? What does `Latitude`/`Longitude` ranking high tell you?

<details><summary>hint</summary>

`permutation_importance(model, X_test, y_test, scoring='r2')`. Look at where `Condition` lands, and remember how it was generated.

</details>

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(hgb, X_test, y_test, n_repeats=5, random_state=42,
                              scoring='r2', n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

imp.plot.barh(figsize=(7, 6), title='Permutation importance (test, R2 drop)')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
display(imp.round(4))

# Condition sits at ~0 (sometimes slightly negative -- permuting pure noise can accidentally
# help). Correct: the model learned to ignore it. A negative permutation importance is a
# useful signal in general: that feature is contributing nothing but variance.
# MedInc dominates, then Latitude/Longitude -- location is the second-biggest driver, and
# because it's an interaction (lat AND long together define "coastal", "Bay Area") only
# the tree models can exploit it. That interaction is most of the linear model's ~0.17 R2
# gap. It also confirms the spatial-autocorrelation worry from Part D: the model leans
# heavily on coordinates, so a random split flatters it.

---

## Debrief — what makes this a *regression* interview

- **Look at the target first.** The 5.0 cap is the whole test here. Candidates who plot the
  target find it in 30 seconds; candidates who go straight to `.fit()` never do.
- **Metrics differ.** No accuracy, no AUC. RMSE vs MAE is a real choice with a real
  argument, and R² is the one number a stakeholder understands.
- **Residual plots are the regression equivalent of a confusion matrix.** Structure in the
  residuals = something the model can't represent.
- **Non-linearity actually pays here** (R² 0.66 → 0.83), unlike Titanic. Knowing *when*
  the fancy model earns its keep is the skill — not always reaching for it.
- **The split is a judgment call.** Spatial autocorrelation means the honest number is
  lower than the one you'll report. Say so before they ask.